# NPA screening of the four white-noise-optimized cycle facets

## Purpose

This notebook performs the quantum screening of the four inequalities produced
by `Cycle_White_Noise_Separator_LP.ipynb`.

It:

1. reads `normalized_coefficients.tsv` exactly;
2. regenerates the corrected 8128 siglocal vertices;
3. reconstructs the four NS targets from classes 27, 29, 43, and 44;
4. audits exact validity, white-noise normalization, target slack, and facet
   dimension for every imported inequality;
5. constructs the corresponding `QuantumNPA.jl` Bell polynomials in canonical
   $p(abc\mid xyz)$ order;
6. computes NPA lower bounds at levels 2, 3, and 4 using tight SCS settings;
7. reports and saves both the **direct NPA inequality value** and the
   **normalized NPA slack**.

For an imported inequality

$$
I(p)=\sum_{a,b,c,x,y,z}c_{abcxyz}p(abc\mid xyz)\geq\beta,
$$

the direct level-$k$ NPA value is

$$
L_k=\min_{p\in\mathcal Q_k}I(p).
$$

The normalized NPA slack is the separate quantity

$$
\Delta_k=L_k-\beta.
$$

The level-4 output table displays $L_4$, $\beta$, and $\Delta_4$ in distinct
columns so that the direct inequality value is never hidden by the normalized
comparison.


## Interpretation before running

The NPA set $\mathcal Q_k$ is an outer relaxation of the quantum set. Therefore
$L_k$ is a lower bound on the true quantum minimum of $I$.

- If $L_k\geq\beta$ by more than the numerical tolerance, the relaxation itself
  obeys the inequality, which excludes a quantum violation.
- If $L_k<\beta$, the relaxation violates the inequality. This makes the
  inequality a candidate for quantum violation, but does **not** yet provide a
  quantum state and measurements. A see-saw calculation is then required.
- Values extremely close to $\beta$ are numerically unresolved.

SCS is a numerical first-order solver. Its returned values are numerical NPA
bounds rather than formally certified SDP bounds.


## How to run

Place this notebook either

- beside `normalized_coefficients.tsv`, or
- beside the directory `cycle_white_noise_lp_results` created by the preceding
  LP notebook.

Then choose **Kernel → Restart Kernel and Run All Cells**. The first code cell
installs only packages missing from the active Julia environment.

Results are written after every completed SDP to
`cycle_white_noise_npa_results`. The principal files are:

- `white_noise_npa_all_levels.tsv`;
- `white_noise_npa_level4_ranking.tsv`;
- `run_metadata.txt`.


In [4]:
# Package bootstrap. Existing package installations are left untouched; only
# packages missing from the active Julia environment are added.
import Pkg

required_packages = [
    "JuMP",
    "MathOptInterface",
    "QuantumNPA",
    "SCS",
]

missing_packages = [
    package for package in required_packages
    if Base.find_package(package) === nothing
]

if !isempty(missing_packages)
    println("Installing missing Julia packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required Julia packages are already available.")
end


All required Julia packages are already available.


In [5]:
using Dates
using JuMP
using LinearAlgebra
using Printf
using QuantumNPA
using SCS
using SHA
using SparseArrays

# Import MathOptInterface directly; do not access it as
# `JuMP.MathOptInterface` under Julia 1.12.
import MathOptInterface as MOI

# -------------------------- USER CONFIGURATION --------------------------

NPA_LEVELS = [2, 3, 4]

# These are the same tight SCS settings used in the previously validated NPA
# screening notebook.
SCS_EPS_ABS = 1e-8
SCS_EPS_REL = 1e-8
SCS_MAX_ITERS = 1_000_000

# Used only to classify and diagnose numerical outputs. It does not alter an
# SDP or an inequality.
NPA_SCREENING_TOLERANCE = 5e-4
NPA_MONOTONICITY_TOLERANCE = 5e-4

RESULTS_DIRECTORY = joinpath(pwd(), "cycle_white_noise_npa_results")

# ------------------------------------------------------------------------

TIGHT_SCS_SOLVER = optimizer_with_attributes(
    SCS.Optimizer,
    "eps_abs" => SCS_EPS_ABS,
    "eps_rel" => SCS_EPS_REL,
    "max_iters" => SCS_MAX_ITERS,
    "verbose" => 0,
)

const CLASSES = (27, 29, 43, 44)
const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

mkpath(RESULTS_DIRECTORY)
println("NPA output directory: ", abspath(RESULTS_DIRECTORY))


NPA output directory: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_white_noise_npa_results


## 1. Canonical probability coordinates

The full behavior is represented by a 64-vector. Input blocks are ordered

`xyz = 000, 001, 010, 011, 100, 101, 110, 111`,

and, inside every block, outputs are ordered

`abc = 000, 001, 010, 011, 100, 101, 110, 111`.

Thus $c$ and $z$ are the fastest-changing indices. The same index function is
used by the vertex generator, TSV audit, target reconstruction, and NPA
operator construction.


In [7]:
"""One-based index of p(a,b,c|x,y,z) in the canonical 64-vector."""
prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)


## 2. Correct deterministic vertices of the two cycle orientations

Each party uses a deterministic binary response function of two binary inputs. There are

$$
2^{2^2}=16
$$

such functions. The integer masks `0:15` encode their four-entry truth tables; `response(mask,u,v)` reads the output assigned to the pair $(u,v)$.

The two communication orientations have different causal dependencies:

| orientation | Alice | Bob | Charlie |
|---|---|---|---|
| clockwise | $a=f_A(x,z)$ | $b=f_B(y,x)$ | $c=f_C(z,y)$ |
| counterclockwise | $a=f_A(x,y)$ | $b=f_B(y,z)$ | $c=f_C(z,x)$ |

Choosing one of 16 response functions independently for each party gives $16^3=4096$ deterministic behaviors per orientation. Every deterministic behavior has exactly one probability equal to 1 in each input block and all other entries equal to 0.

The polytope used by the LP is the convex hull of the **union** of the two orientation lists. Their intersection contains the 64 fully local deterministic behaviors, so removing duplicate rows gives

$$
4096+4096-64=8128
$$

distinct vertices. Removing duplicates does not change the convex hull or any LP optimum, but it exposes the correct combinatorics and avoids redundant constraints.

This cell performs structural checks before any inequality is computed: binary entries, one selected output per input, expected vertex counts, intersection size, and affine dimensions. The individual orientation polytopes have affine dimension 44; their corrected union has affine dimension 50. In particular, the jump from 44 to 50 is a strong diagnostic that the second orientation was not accidentally generated as a copy of the first.


In [9]:

"""
    response(mask, u, v)

Evaluate one of the 16 deterministic binary functions of two binary inputs.
Bit `2u + v` of `mask` stores the output for the input pair `(u,v)`.
"""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""
    deterministic_vertex(fA, fB, fC, orientation)

Construct one deterministic 64-component behavior in canonical coordinates.

For `:clockwise`:
    a = fA(x,z),  b = fB(y,x),  c = fC(z,y).

For `:counterclockwise`:
    a = fA(x,y),  b = fB(y,z),  c = fC(z,x).
"""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one cycle orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function affine_dimension(points; atol=1e-9)
    n = size(points, 1)
    n == 0 && return -1
    n == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64

    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)

        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end

    return true
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# `unique(...; dims=1)` removes the 64 local deterministic behaviors that occur
# in both orientations. Keeping duplicates would not change the LP, but it would
# obscure the actual vertex count and the geometry.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise, 1) == 4096
@assert size(vertices_counterclockwise, 1) == 4096
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle, 1) == 8128

@assert affine_dimension(vertices_clockwise) == 44
@assert affine_dimension(vertices_counterclockwise) == 44
@assert affine_dimension(vertices_cycle) == 50

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))
println("  affine dimension:          ", affine_dimension(vertices_cycle))




Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128
  affine dimension:          50


## 3. Published representatives of NS classes 27, 29, 43, and 44

The four targets are entered through the 26 correlators of a binary tripartite nonsignalling behavior. Their stored order is

$$
\begin{aligned}
&(A_0,A_1;\ B_0,B_1;\ C_0,C_1;\\
&AB_{00},AB_{01},AB_{10},AB_{11};\\
&AC_{00},AC_{01},AC_{10},AC_{11};\\
&BC_{00},BC_{01},BC_{10},BC_{11};\\
&ABC_{000},ABC_{001},ABC_{010},ABC_{011},
ABC_{100},ABC_{101},ABC_{110},ABC_{111}).
\end{aligned}
$$

These representatives are those reported in Table 1 of S. Pironio, J.-D. Bancal, and V. Scarani, *Extremal correlations of the tripartite no-signaling polytope*, J. Phys. A **44**, 065303 (2011), arXiv:1101.2477.

To convert correlators into probabilities, define $s_a=(-1)^a$, and analogously for $b$ and $c$. Then

$$
\begin{aligned}
p(abc\mid xyz)=\frac18[&1+s_aA_x+s_bB_y+s_cC_z
+s_as_bAB_{xy}+s_as_cAC_{xz}\\
&+s_bs_cBC_{yz}+s_as_bs_cABC_{xyz}].
\end{aligned}
$$

The implementation uses Julia rational numbers such as `1//3` throughout this conversion. Consequently, positivity, normalization, nonsignalling, and the later evaluations of rationalized inequalities can be checked exactly rather than up to floating-point tolerance.


In [11]:
#
# Correlator order:
#
#   A0,A1, B0,B1, C0,C1,
#   AB00,AB01,AB10,AB11,
#   AC00,AC01,AC10,AC11,
#   BC00,BC01,BC10,BC11,
#   ABC000,ABC001,ABC010,ABC011,ABC100,ABC101,ABC110,ABC111.
#
# These representatives agree with Table 1 of:
# S. Pironio, J.-D. Bancal, and V. Scarani,
# "Extremal correlations of the tripartite no-signaling polytope",
# J. Phys. A 44, 065303 (2011), arXiv:1101.2477.

const NS_CORRELATORS = Dict(
    27 => Rational{Int}[
        1//2, 0, 0, 0, 0, 0,
        1//2, 1//2, 0, 0,
        1//2, 1//2, 0, 0,
        0, 0, 0, 0,
        -1//2, -1//2, -1//2, -1//2, 1, 1, 1, -1,
    ],
    29 => Rational{Int}[
        1//3, 1//3, 1//3, 1//3, 1//3, 1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1//3, 1//3, 1//3, -1, 1//3, -1, -1, -1,
    ],
    43 => Rational{Int}[
        1//7, 1//7, 1//7, 1//7, 1//7, 1//7,
        3//7, -1//7, -1//7, -1//7,
        3//7, -1//7, -1//7, -1//7,
        -1//7, -1//7, -1//7, -1//7,
        1//7, 5//7, 5//7, -1, -1, -1, -1, -1,
    ],
    44 => Rational{Int}[
        0, 0, 0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, -1,
    ],
)

outcome_sign(bit::Int) = bit == 0 ? 1 : -1

"""Convert the 26 NS correlators to the canonical 64 probabilities exactly."""
function correlators_to_probabilities(correlators)
    @assert length(correlators) == 26
    q = Rational{Int}.(correlators)
    probabilities = fill(0//1, 64)

    A = q[1:2]
    B = q[3:4]
    C = q[5:6]
    AB = q[7:10]
    AC = q[11:14]
    BC = q[15:18]
    ABC = q[19:26]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        sa = outcome_sign(a)
        sb = outcome_sign(b)
        sc = outcome_sign(c)

        probabilities[prob_index(a, b, c, x, y, z)] = (
            1 +
            sa*A[x + 1] +
            sb*B[y + 1] +
            sc*C[z + 1] +
            sa*sb*AB[2*x + y + 1] +
            sa*sc*AC[2*x + z + 1] +
            sb*sc*BC[2*y + z + 1] +
            sa*sb*sc*ABC[4*x + 2*y + z + 1]
        ) / 8
    end

    return probabilities
end

const NS_TARGETS = Dict(
    class => correlators_to_probabilities(NS_CORRELATORS[class])
    for class in CLASSES
)




Dict{Int64, Vector{Rational{Int64}}} with 4 entries:
  29 => [2//3, 0, 0, 0, 0, 0, 0, 1//3, 1//3, 1//3  …  0, 0, 0, 1//3, 1//3, 0, 1…
  43 => [2//7, 1//7, 1//7, 0, 0, 1//7, 1//7, 1//7, 2//7, 1//7  …  0, 1//7, 0, 2…
  27 => [1//4, 1//4, 1//4, 0, 0, 0, 0, 1//4, 1//4, 1//4  …  1//4, 0, 0, 1//4, 1…
  44 => [1//4, 0, 0, 1//4, 0, 1//4, 1//4, 0, 1//4, 0  …  1//4, 0, 0, 1//4, 1//4…

## 4. Exact target checks: positivity, normalization, nonsignalling, and extremality

Before using a target in an LP, this cell verifies the defining probabilistic and nonsignalling conditions directly in the 64-coordinate representation:

- every probability is nonnegative;
- the eight probabilities for each fixed input triple $(x,y,z)$ sum to 1;
- the $AB$ marginal is independent of Charlie's input $z$;
- the $AC$ marginal is independent of Bob's input $y$;
- the $BC$ marginal is independent of Alice's input $x$.

It then independently checks that each representative is a **vertex** of the nonsignalling polytope. The normalization and nonsignalling equalities are assembled as a (redundant) $56\times64$ matrix. Its rank is 38, hence the nonsignalling affine space has dimension

$$
64-38=26.
$$

At a given target, every zero probability saturates one positivity inequality. The target is extremal when the equality rows together with these active positivity rows have rank 64: there is then no nonzero feasible direction that keeps all active constraints saturated. The printed `active rank = 64 (NS vertex)` is therefore a geometric check, not merely a lookup of the class label.

All these tests use exact target probabilities. A failed assertion indicates an inconsistent representative, correlator order, probability order, or nonsignalling implementation and should be investigated before proceeding.


In [13]:

function validate_nonsignalling_behavior(probabilities)
    @assert length(probabilities) == 64
    @assert all(probability -> probability >= 0, probabilities)

    # Normalization for every input triple.
    for (x, y, z) in TRIPLES
        @assert sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        ) == 1
    end

    # AB marginal independent of z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        @assert lhs == rhs
    end

    # AC marginal independent of y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        @assert lhs == rhs
    end

    # BC marginal independent of x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        @assert lhs == rhs
    end

    return true
end

"""
Return a (redundant) matrix for normalization and tripartite NS equalities.
Its rank must be 38, leaving the familiar 26-dimensional NS affine space.
"""
function nonsignalling_equality_matrix()
    equalities = zeros(Float64, 56, 64)
    row = 0

    for (x, y, z) in TRIPLES
        row += 1
        for (a, b, c) in TRIPLES
            equalities[row, prob_index(a, b, c, x, y, z)] = 1
        end
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        row += 1
        for c in 0:1
            equalities[row, prob_index(a, b, c, x, y, 0)] += 1
            equalities[row, prob_index(a, b, c, x, y, 1)] -= 1
        end
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        row += 1
        for b in 0:1
            equalities[row, prob_index(a, b, c, x, 0, z)] += 1
            equalities[row, prob_index(a, b, c, x, 1, z)] -= 1
        end
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        row += 1
        for a in 0:1
            equalities[row, prob_index(a, b, c, 0, y, z)] += 1
            equalities[row, prob_index(a, b, c, 1, y, z)] -= 1
        end
    end

    @assert row == 56
    return equalities
end

const NS_EQUALITIES = nonsignalling_equality_matrix()
@assert rank(NS_EQUALITIES; atol=1e-9, rtol=0.0) == 38

"""
Check extremality in the NS polytope using the rank of the NS equalities plus
the positivity inequalities saturated by the target.
"""
function ns_active_rank(probabilities)
    zero_coordinates = findall(iszero, probabilities)
    coordinate_rows = Matrix{Float64}(I, 64, 64)[zero_coordinates, :]
    active_matrix = vcat(NS_EQUALITIES, coordinate_rows)
    return rank(active_matrix; atol=1e-9, rtol=0.0)
end

println("\nTarget checks")
for class in CLASSES
    target = NS_TARGETS[class]
    validate_nonsignalling_behavior(target)
    active_rank = ns_active_rank(target)
    @assert active_rank == 64

    @printf(
        "  class %2d: support = %2d, active rank = %2d (NS vertex)\n",
        class,
        count(value -> !iszero(value), target),
        active_rank,
    )
end





Target checks
  class 27: support = 32, active rank = 64 (NS vertex)
  class 29: support = 23, active rank = 64 (NS vertex)
  class 43: support = 36, active rank = 64 (NS vertex)
  class 44: support = 32, active rank = 64 (NS vertex)


## 5. Read the normalized inequalities exactly

The input is searched for in two places:

1. beside this notebook;
2. inside `cycle_white_noise_lp_results`, where the preceding notebook writes
   it by default.

Every entry is parsed as `Rational{BigInt}`. No coefficient is read through
floating point during the exact classical audit.


In [15]:
function locate_input_file(filename)
    possible_paths = [
        joinpath(pwd(), filename),
        joinpath(pwd(), "cycle_white_noise_lp_results", filename),
    ]

    for path in possible_paths
        isfile(path) && return path
    end

    error(
        "Could not find $filename. Put it beside the notebook or retain " *
        "the cycle_white_noise_lp_results directory beside the notebook.",
    )
end

"""Parse strings such as `-8//45`, `3`, or `0//1` exactly."""
function parse_exact_rational(text)
    pieces = split(strip(text), "//")
    if length(pieces) == 1
        return parse(BigInt, pieces[1]) // BigInt(1)
    elseif length(pieces) == 2
        return parse(BigInt, pieces[1]) // parse(BigInt, pieces[2])
    else
        error("Invalid rational entry: $text")
    end
end

function read_normalized_coefficients(path)
    lines = readlines(path)
    length(lines) >= 2 || error("The coefficient TSV contains no data rows")

    expected_header = vcat(
        ["class", "beta"],
        ["c_$index" for index in 1:64],
    )
    header = split(lines[1], '\t'; keepempty=true)
    header == expected_header || error(
        "Unexpected coefficient TSV header. Expected class, beta, and c_1 through c_64.",
    )

    candidates = Dict{Int, Any}()
    for line in lines[2:end]
        isempty(strip(line)) && continue
        fields = split(line, '\t'; keepempty=true)
        length(fields) == 66 || error(
            "Each coefficient row must contain class, beta, and 64 coefficients",
        )

        class = parse(Int, fields[1])
        haskey(candidates, class) && error("Duplicate class $class in coefficient TSV")

        beta = parse_exact_rational(fields[2])
        coefficients = parse_exact_rational.(fields[3:end])
        length(coefficients) == 64 || error("Class $class does not have 64 coefficients")

        candidates[class] = (
            class=class,
            beta=beta,
            coefficients=coefficients,
        )
    end

    Set(keys(candidates)) == Set(CLASSES) || error(
        "The coefficient TSV must contain exactly classes 27, 29, 43, and 44",
    )
    return candidates
end

COEFFICIENT_INPUT_PATH = locate_input_file("normalized_coefficients.tsv")
IMPORTED_CANDIDATES = read_normalized_coefficients(COEFFICIENT_INPUT_PATH)

println("Coefficient input: ", abspath(COEFFICIENT_INPUT_PATH))
println("Imported classes: ", sort(collect(keys(IMPORTED_CANDIDATES))))


Coefficient input: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_white_noise_lp_results\normalized_coefficients.tsv
Imported classes: [27, 29, 43, 44]


## 6. Independent exact audit of the imported inequalities

For every imported inequality, this section verifies:

- validity on all 8128 siglocal vertices;
- minimum slack zero in each orientation and their union;
- uniform-behavior slack exactly one;
- the expected exact slack at its NS target;
- a 49-dimensional saturating face inside the 50-dimensional polytope.

Only the affine-rank calculation uses floating-point linear algebra; all
inequality values, minima, and saturation decisions are exact rational
calculations.


In [17]:
const UNIFORM_BEHAVIOUR_BIG = fill(BigInt(1) // BigInt(8), 64)
const EXPECTED_TARGET_SLACKS_BIG = Dict(
    27 => -BigInt(1) // BigInt(11),
    29 => -BigInt(8) // BigInt(45),
    43 => -BigInt(2) // BigInt(21),
    44 => -BigInt(1) // BigInt(3),
)

to_big_rational(value::Rational) =
    BigInt(numerator(value)) // BigInt(denominator(value))

function exact_value(coefficients, behavior)
    return sum(
        coefficients[index] * to_big_rational(behavior[index])
        for index in eachindex(coefficients)
    )
end

function exact_value(coefficients, behavior::AbstractVector{<:Integer})
    return sum(
        coefficients[index] * BigInt(behavior[index])
        for index in eachindex(coefficients)
    )
end

audited_candidates = Dict{Int, Any}()
polytope_dimension = affine_dimension(vertices_cycle)
@assert polytope_dimension == 50

println("\nExact imported-inequality audit")
println("class  beta      target-slack  sat. vertices  face-dim  facet")

for class in CLASSES
    candidate = IMPORTED_CANDIDATES[class]
    coefficients = candidate.coefficients
    beta = candidate.beta

    cycle_slacks = [
        exact_value(coefficients, collect(@view vertices_cycle[row, :])) - beta
        for row in axes(vertices_cycle, 1)
    ]
    clockwise_slacks = [
        exact_value(coefficients, collect(@view vertices_clockwise[row, :])) - beta
        for row in axes(vertices_clockwise, 1)
    ]
    counterclockwise_slacks = [
        exact_value(coefficients, collect(@view vertices_counterclockwise[row, :])) - beta
        for row in axes(vertices_counterclockwise, 1)
    ]

    uniform_slack = exact_value(coefficients, UNIFORM_BEHAVIOUR_BIG) - beta
    target_big = to_big_rational.(NS_TARGETS[class])
    target_slack = exact_value(coefficients, target_big) - beta

    @assert minimum(cycle_slacks) == 0
    @assert minimum(clockwise_slacks) == 0
    @assert minimum(counterclockwise_slacks) == 0
    @assert all(slack -> slack >= 0, cycle_slacks)
    @assert uniform_slack == 1
    @assert target_slack == EXPECTED_TARGET_SLACKS_BIG[class]

    saturating_indices = findall(iszero, cycle_slacks)
    face_dimension = affine_dimension(vertices_cycle[saturating_indices, :])
    is_facet = face_dimension == polytope_dimension - 1

    @assert face_dimension == 49
    @assert is_facet

    audited_candidates[class] = merge(candidate, (
        uniform_slack=uniform_slack,
        target_slack=target_slack,
        saturation_count=length(saturating_indices),
        face_dimension=face_dimension,
        is_facet=is_facet,
    ))

    @printf(
        "%5d  %8s  %12s  %13d  %8d  %s\n",
        class,
        string(beta),
        string(target_slack),
        length(saturating_indices),
        face_dimension,
        string(is_facet),
    )
end



Exact imported-inequality audit
class  beta      target-slack  sat. vertices  face-dim  facet
   27     -1//1        -1//11            119        49  true
   29   -32//15        -8//45            270        49  true
   43     -4//3        -2//21            508        49  true
   44      1//3         -1//3            508        49  true


## 7. Build the four `QuantumNPA.jl` Bell polynomials

For each canonical coordinate, the probability operator is

$$
P^A_{a\mid x}P^B_{b\mid y}P^C_{c\mid z}.
$$

The imported coefficient vector is mapped directly through `prob_index`; no
manually reordered coefficient matrix is used.

The compatibility method below is the same one used successfully in the
previous tight-SCS notebook. It is installed only if the active
`QuantumNPA`/`JuMP` combination does not already provide it.


In [19]:
function install_quantumnpa_jump_compatibility!()
    signature = Tuple{
        SparseMatrixCSC{Float64, Int},
        Symmetric{JuMP.VariableRef, Matrix{JuMP.VariableRef}},
    }

    if !isdefined(QuantumNPA, :dot) ||
       !hasmethod(getfield(QuantumNPA, :dot), signature)
        @eval QuantumNPA begin
            import JuMP: VariableRef
            import LinearAlgebra: Symmetric
            import SparseArrays: SparseMatrixCSC

            function dot(
                A::SparseMatrixCSC{Float64, Int},
                B::Symmetric{VariableRef, Matrix{VariableRef}},
            )
                return sum(
                    A[i, j] * B[j, i]
                    for i in 1:size(A, 1), j in 1:size(A, 2)
                )
            end
        end
        println("Installed the QuantumNPA/JuMP sparse-dot compatibility method.")
    else
        println("QuantumNPA/JuMP sparse-dot compatibility method already available.")
    end
end

install_quantumnpa_jump_compatibility!()

"""Create the 64 probability operators in canonical vector order."""
function build_probability_operators()
    PA = projector(1, 1:2, 1:2, full=true)
    PB = projector(2, 1:2, 1:2, full=true)
    PC = projector(3, 1:2, 1:2, full=true)

    operators = Vector{Any}(undef, 64)
    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        operators[prob_index(a, b, c, x, y, z)] =
            PA[a + 1, x + 1] * PB[b + 1, y + 1] * PC[c + 1, z + 1]
    end
    return operators
end

"""Build one QuantumNPA polynomial from an exact coefficient vector."""
function build_bell_expression(coefficients, probability_operators)
    length(coefficients) == length(probability_operators) == 64 ||
        error("A Bell expression requires exactly 64 coefficients/operators")

    expression = nothing
    for index in eachindex(coefficients)
        iszero(coefficients[index]) && continue
        term = Float64(coefficients[index]) * probability_operators[index]
        expression = expression === nothing ? term : expression + term
    end

    expression === nothing && error("The Bell functional is identically zero")
    return expression
end

PROBABILITY_OPERATORS = build_probability_operators()
BELL_EXPRESSIONS = Dict(
    class => build_bell_expression(
        audited_candidates[class].coefficients,
        PROBABILITY_OPERATORS,
    )
    for class in CLASSES
)

@assert length(PROBABILITY_OPERATORS) == 64
println("Constructed all four Bell expressions in canonical p(abc|xyz) order.")


QuantumNPA/JuMP sparse-dot compatibility method already available.
Constructed all four Bell expressions in canonical p(abc|xyz) order.


## 8. Tight-SCS NPA scan at levels 2, 3, and 4

For every class and level, the saved row contains:

- `npa_value`: the direct NPA value $L_k=\min_{\mathcal Q_k}I(p)$;
- `beta`: the siglocal bound of that same normalized inequality;
- `normalized_npa_slack`: $\Delta_k=L_k-\beta$;
- `beta_minus_npa_value`: $\beta-L_k=-\Delta_k$;
- `npa_noise_upper_bound`: the white-noise robustness allowed by the outer
  relaxation when $\Delta_k<0$.

Thus `npa_value`, especially its level-4 value, is explicitly retained and is
not replaced by the normalized slack.


In [21]:
ALL_LEVELS_PATH = joinpath(RESULTS_DIRECTORY, "white_noise_npa_all_levels.tsv")
LEVEL4_PATH = joinpath(RESULTS_DIRECTORY, "white_noise_npa_level4_ranking.tsv")

function screening_label(slack; tolerance=NPA_SCREENING_TOLERANCE)
    if slack < -tolerance
        return "npa_candidate"
    elseif slack > tolerance
        return "npa_no_violation"
    else
        return "numerically_unresolved"
    end
end

function write_all_levels(path, rows)
    open(path, "w") do io
        println(io, join([
            "class",
            "level",
            "beta",
            "npa_value",
            "normalized_npa_slack",
            "beta_minus_npa_value",
            "npa_noise_upper_bound",
            "classification",
            "elapsed_seconds",
        ], "\t"))

        for row in rows
            println(io, join([
                string(row.class),
                string(row.level),
                @sprintf("%.16g", row.beta),
                @sprintf("%.16g", row.npa_value),
                @sprintf("%.16g", row.normalized_npa_slack),
                @sprintf("%.16g", row.beta_minus_npa_value),
                @sprintf("%.16g", row.npa_noise_upper_bound),
                row.classification,
                @sprintf("%.6f", row.elapsed_seconds),
            ], "\t"))
        end
    end
end

npa_rows = NamedTuple[]

for class in CLASSES
    candidate = audited_candidates[class]
    expression = BELL_EXPRESSIONS[class]
    beta = Float64(candidate.beta)

    for level in NPA_LEVELS
        println("Class $class: starting NPA level $level")
        timed_result = @timed npa_min(
            expression,
            level;
            solver=TIGHT_SCS_SOLVER,
        )

        npa_value = Float64(timed_result.value)
        isfinite(npa_value) || error(
            "QuantumNPA returned non-finite value $npa_value for class $class, level $level",
        )

        normalized_npa_slack = npa_value - beta
        beta_minus_npa_value = beta - npa_value
        npa_noise_upper_bound = normalized_npa_slack < 0 ?
            -normalized_npa_slack / (1 - normalized_npa_slack) : 0.0
        classification = screening_label(normalized_npa_slack)

        push!(npa_rows, (
            class=class,
            level=level,
            beta=beta,
            npa_value=npa_value,
            normalized_npa_slack=normalized_npa_slack,
            beta_minus_npa_value=beta_minus_npa_value,
            npa_noise_upper_bound=npa_noise_upper_bound,
            classification=classification,
            elapsed_seconds=timed_result.time,
        ))

        # Rewrite the table after every successful SDP. If the run is later
        # interrupted, all completed values remain on disk.
        write_all_levels(ALL_LEVELS_PATH, npa_rows)

        @printf(
            "  direct NPA value L_%d = %.12f; beta = %.12f; slack = %.12f; %.2f s\n",
            level,
            npa_value,
            beta,
            normalized_npa_slack,
            timed_result.time,
        )

        GC.gc()
    end
end

# Exact hierarchy values for minimization are nondecreasing. Report numerical
# inversions larger than the declared tolerance.
for class in CLASSES
    class_rows = sort(
        [row for row in npa_rows if row.class == class];
        by=row -> row.level,
    )

    for index in 2:length(class_rows)
        previous = class_rows[index - 1]
        current = class_rows[index]
        if current.npa_value < previous.npa_value - NPA_MONOTONICITY_TOLERANCE
            @warn "NPA values are not numerically monotone" class previous current
        end
    end
end

println("Saved all-level NPA results to: ", abspath(ALL_LEVELS_PATH))


Class 27: starting NPA level 2
  direct NPA value L_2 = -0.999999999768; beta = -1.000000000000; slack = 0.000000000232; 22.64 s
Class 27: starting NPA level 3
  direct NPA value L_3 = -1.000000000140; beta = -1.000000000000; slack = -0.000000000140; 0.80 s
Class 27: starting NPA level 4
  direct NPA value L_4 = -0.999999999828; beta = -1.000000000000; slack = 0.000000000172; 3.03 s
Class 29: starting NPA level 2
  direct NPA value L_2 = -2.144789444731; beta = -2.133333333333; slack = -0.011456111398; 13.82 s
Class 29: starting NPA level 3
  direct NPA value L_3 = -2.136293446388; beta = -2.133333333333; slack = -0.002960113055; 926.61 s
Class 29: starting NPA level 4
  direct NPA value L_4 = -2.136293398937; beta = -2.133333333333; slack = -0.002960065603; 95.80 s
Class 43: starting NPA level 2
  direct NPA value L_2 = -1.333333328468; beta = -1.333333333333; slack = 0.000000004865; 13.32 s
Class 43: starting NPA level 3
  direct NPA value L_3 = -1.333333334943; beta = -1.33333333333

## 9. Level-4 direct values and ranking

This is the principal output requested for the project. The table prints the
direct level-4 NPA inequality value $L_4$ first, followed by the inequality
bound $\beta$ and the derived normalized slack $L_4-\beta$.

Candidates are ranked by normalized slack because every inequality has the
same white-noise normalization $I(u)-\beta=1$. A raw ratio such as
$L_4/\beta$ is not appropriate here: some bounds are negative, and the ratio is
not invariant under equivalent constant shifts of a Bell functional.


In [23]:
level4_rows = sort(
    [row for row in npa_rows if row.level == 4];
    by=row -> row.normalized_npa_slack,
)

length(level4_rows) == length(CLASSES) || error(
    "Expected one completed level-4 result for each class",
)

println("\nLevel-4 NPA results, ranked by normalized NPA slack")
println("class    direct NPA value L4          beta       L4-beta      beta-L4  classification")

for row in level4_rows
    @printf(
        "%5d  %21.12f  %12.8f  %12.8f  %12.8f  %s\n",
        row.class,
        row.npa_value,
        row.beta,
        row.normalized_npa_slack,
        row.beta_minus_npa_value,
        row.classification,
    )
end

open(LEVEL4_PATH, "w") do io
    println(io, join([
        "rank",
        "class",
        "level4_npa_value",
        "beta",
        "normalized_npa_slack",
        "beta_minus_npa_value",
        "npa_noise_upper_bound",
        "classification",
    ], "\t"))

    for (rank_index, row) in enumerate(level4_rows)
        println(io, join([
            string(rank_index),
            string(row.class),
            @sprintf("%.16g", row.npa_value),
            @sprintf("%.16g", row.beta),
            @sprintf("%.16g", row.normalized_npa_slack),
            @sprintf("%.16g", row.beta_minus_npa_value),
            @sprintf("%.16g", row.npa_noise_upper_bound),
            row.classification,
        ], "\t"))
    end
end

println("Saved level-4 ranking to: ", abspath(LEVEL4_PATH))



Level-4 NPA results, ranked by normalized NPA slack
class    direct NPA value L4          beta       L4-beta      beta-L4  classification
   29        -2.136293398937   -2.13333333   -0.00296007    0.00296007  npa_candidate
   43        -1.333333334268   -1.33333333   -0.00000000    0.00000000  numerically_unresolved
   27        -0.999999999828   -1.00000000    0.00000000   -0.00000000  numerically_unresolved
   44         0.333333346066    0.33333333    0.00000001   -0.00000001  numerically_unresolved
Saved level-4 ranking to: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_white_noise_npa_results\white_noise_npa_level4_ranking.tsv


## 10. Reproducibility metadata

The final cell records the Julia version, input-file checksum, solver settings,
and exact imported inequality data. Send back the two result TSV files and the
metadata file for validation.


In [25]:
METADATA_PATH = joinpath(RESULTS_DIRECTORY, "run_metadata.txt")
input_checksum = bytes2hex(sha256(read(COEFFICIENT_INPUT_PATH)))

open(METADATA_PATH, "w") do io
    println(io, "White-noise facet NPA screening")
    println(io, "timestamp = ", now())
    println(io, "julia_version = ", VERSION)
    println(io, "coefficient_input = ", abspath(COEFFICIENT_INPUT_PATH))
    println(io, "coefficient_sha256 = ", input_checksum)
    println(io, "npa_levels = ", join(NPA_LEVELS, ","))
    println(io, "scs_eps_abs = ", SCS_EPS_ABS)
    println(io, "scs_eps_rel = ", SCS_EPS_REL)
    println(io, "scs_max_iters = ", SCS_MAX_ITERS)
    println(io, "screening_tolerance = ", NPA_SCREENING_TOLERANCE)
    println(io, "monotonicity_tolerance = ", NPA_MONOTONICITY_TOLERANCE)

    for class in CLASSES
        candidate = audited_candidates[class]
        println(io, "class_$(class)_beta = ", candidate.beta)
        println(io, "class_$(class)_target_slack = ", candidate.target_slack)
        println(io, "class_$(class)_saturation_count = ", candidate.saturation_count)
        println(io, "class_$(class)_face_dimension = ", candidate.face_dimension)
        println(io, "class_$(class)_is_facet = ", candidate.is_facet)
    end
end

println("Saved run metadata to: ", abspath(METADATA_PATH))


Saved run metadata to: C:\Users\josen\OneDrive\Desktop\Work\Post-doc_Recife\alternative_projects\Rabello's_project\CV_dropbox_content\Projeto_Ze_Gabrielzinho_Rabelo_Planeta\Jupiter\cycle_white_noise_npa_results\run_metadata.txt


## What to inspect after the run

For each class, compare the printed direct value $L_4$ with its printed bound
$\beta$:

- $L_4<\beta$: NPA candidate; run a see-saw next;
- $L_4>\beta$: no quantum violation is allowed by that NPA relaxation, subject
  to the numerical reliability of the SCS result;
- $L_4\approx\beta$: rerun or audit numerical precision before drawing a
  conclusion.

The ranking by $L_4-\beta$ is useful for comparison, but the file retains the
direct $L_4$ value in its own column exactly as requested.
